# Stored Procedures
A Stored Procedure in SQL is a prepared SQL code that you can save and reuse over and over again. Instead of writing the same complex query multiple times, you save it as a stored procedure and simply call it whenever you need to execute it. It is mainly used to improve database performance, simplify maintenance, and enhance security by restricting direct access to your tables.

In [3]:
import os
from dotenv import load_dotenv

import pandas as pd
import sqlalchemy

In [4]:
load_dotenv()

db_host = os.environ.get("db_host")
db_user = os.environ.get("db_user")
db_password = os.environ.get("db_password")

In [5]:
engine = sqlalchemy.create_engine(f"mysql+pymysql://{db_user}:{db_password}@{db_host}:3306/sql_invoicing")

In [6]:
pd.read_sql("SHOW TABLES", con= engine)

,Tables_in_sql_invoicing
0,clients
1,clients_balance
2,invoices
3,payment_methods
4,payments


## Creating a Stored Procedure

In [7]:
query = sqlalchemy.text("""
CREATE PROCEDURE get_clients()
BEGIN
    SELECT * FROM clients;
END
""")


# but in MySQL we should create a procedure like this:
'''
DELIMITER $$
CREATE PROCEDURE get_clients()
BEGIN
    SELECT * FROM clients;
END$$

DELIMITER ;
'''



with engine.begin() as conn:
    conn.execute(query)

We can call a procedure like this:

In [8]:
query = """
call get_clients()
"""

pd.read_sql(query, con= engine)

,client_id,name,address,city,state,phone
0,1,Vinte,3 Nevada Parkway,Syracuse,NY,315-252-7305
1,2,Myworks,34267 Glendale Parkway,Huntington,WV,304-659-1170
2,3,Yadel,096 Pawling Parkway,San Francisco,CA,415-144-6037
3,4,Kwideo,81674 Westerfield Circle,Waco,TX,254-750-0784
4,5,Topiclounge,0863 Farmco Road,Portland,OR,971-888-9129


EXERCISE:

In [10]:
query = sqlalchemy.text("""
create procedure get_invoices_with_balance()
begin
select *
from invoices
where invoice_total - payment_total > 0;
end
""")



with engine.begin() as conn:
    conn.execute(query)

## Dropping Stored Proceures

In [11]:
query = sqlalchemy.text("""
DROP PROCEDURE IF EXISTS get_clients
""")



with engine.begin() as conn:
    conn.execute(query)

## Parameters

In [13]:
query = sqlalchemy.text("""
create procedure get_clients_by_state(state char(2))
begin
	select * 
    from clients c
    where c.state = state;
end
""")



with engine.begin() as conn:
    conn.execute(query)

## Parameters with Default Value

In [15]:
query = sqlalchemy.text("""
create procedure get_clients_by_state(state char(2))
begin
	select * 
    from clients c
    where c.state = IFNULL(state, c.state);
end
""")



with engine.begin() as conn:
    conn.execute(query)

## Parameter Validation

In [17]:
# Search about sqlstate code for different types of error

query = sqlalchemy.text("""
CREATE PROCEDURE make_payment(
	invoice_id INT,
    payment_amount DECIMAL(9, 2),
    payment_date DATE
)
BEGIN
    IF payment_amount <= 0 then
        signal sqlstate '22003'
            set message_text = 'invalid payment amount';
    END IF;
            
	UPDATE invoices i
    SET i.payment_total = payment_amount,
		i.payment_date =  payment_date 
	WHERE i.invoice_id = invoice_id;
END
""")



with engine.begin() as conn:
    conn.execute(query)

## Output Parameters

In [19]:
query = sqlalchemy.text("""
create procedure get_unpaid_invoices(
client_id INT,
OUT invoices_total DECIMAL(9,2)
)

begin
    SELECT SUM(invoice_total)
    INTO invoices_total
    from invoices i
    where i.client_id = client_id;

end
""")

with engine.begin() as conn:
    conn.execute(query)

## Variables

In [23]:
# Local Variable
"""
declare total_invoices decimal(9,2);
select sum(invoice) into total_invoices;
"""

'\ndeclare total_invoices decimal(9,2);\nselect sum(invoice) into total_invoices;\n'

In [21]:
# User Variable
"""
set @PI = 3.14;
"""

'\nset @PI = 3.14;\n'

## Functions

In SQL, a Function must return a value and can be used inside SQL statements such as `SELECT`, `WHERE`, or `ORDER BY`. It is typically used for calculations or data transformations.

A Procedure does not have to return a value and is executed using a `CALL` statement. Procedures are usually used to perform actions, such as inserting, updating, deleting data, or executing multiple SQL statements as a single task.

In [28]:
query = sqlalchemy.text("""
CREATE FUNCTION get_risk_factor()
RETURNS INTEGER
DETERMINISTIC
NO SQL
BEGIN
    RETURN 1;
END
""")

with engine.begin() as conn:
    conn.execute(query)